# 3. Particiones y preprocesamiento dentro del entrenamiento
**Objetivo:** separar aprender de evaluar. La CV externa reserva casos para medir desempeño;
la CV interna, contenida en el entrenamiento externo, permite ajustar hiperparámetros y familia.
Si hay réplicas o configuraciones dependientes, todos sus registros permanecen en un mismo grupo.

Completar `config/model_config.yml` solo después de la curaduría. El número de folds se decide
según observaciones y grupos efectivos. No se fija aquí un tamaño de muestra.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ingesta.py').is_file()
             and (p / 'config' / 'data_schema.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde el repositorio o notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.experimento import read_config, inspect_plan, load_model_data
from src.particiones import nested_plan
from src.modelos import candidate_specs

config, config_hash = read_config(ROOT / 'config' / 'model_config.yml')
inspect_plan(config)  # Inspección: nunca llama fit


In [ ]:
plan = None
if config['data']['model_table'] and config['protocol']['reviewed']:
    X, y, trace, groups, quality, _, _ = load_model_data(config)
    plan = nested_plan(X, config['protocol'], groups)
    display(pd.DataFrame([{'fold': p['fold'], 'train': len(p['train']),
                           'evaluación': len(p['test']), 'folds_internos': len(p['inner'])}
                          for p in plan]))
else:
    print('Faltan datos reales o revisión del protocolo; no se construyen particiones.')


## Transformaciones aprendidas
Cada candidato contiene un `Pipeline`: imputación de medianas → eliminación de constantes →
escalamiento cuando corresponde → regresor. Al ajustar cada fold, se calculan parámetros
exclusivamente con su entrenamiento. La revisión rechaza columnas completamente ausentes en
algún entrenamiento: no existe una mediana verificable en ese caso.

El escalamiento no altera la unidad final del target; los árboles no lo requieren. Las fórmulas
geométricas fijas se documentan en curaduría. La selección supervisada adicional y representaciones
alternativas requerirán incorporarse a la búsqueda interna, nunca escogerse con el error externo.


In [ ]:
candidates = candidate_specs(config)
{name: pipeline for name, (pipeline, grid) in candidates.items()}  # Sin fit


## Decisión metodológica
Registrar justificación de independencia/grupos, unidad de generalización, folds y seed.
Revisar que un grupo no se reparta entre train y test. La función guarda índices globales
y ordinales de origen al ejecutar el experimento. Continuar con `Baseline_basico.ipynb`.
